# Phase 1: Data Infrastructure & Empirical Diagnostics  
## Notebook 01_02 — Brownian Bridge Imputation for Missing Financial Time Series

### Research question

How can Brownian bridge interpolation be used to impute missing observations in financial time series while preserving endpoint consistency and making uncertainty explicit?

This notebook builds a practical imputation layer for missing price data:

1. simulate a clean synthetic asset price path;
2. introduce controlled missing-data gaps;
3. compare forward fill, linear interpolation, and Brownian bridge imputation;
4. derive the Brownian bridge conditional mean and variance;
5. impute missing **log prices** rather than raw prices;
6. generate multiple plausible bridge paths, not just one filled series;
7. evaluate imputation error on deliberately masked observations;
8. discuss why Brownian bridge imputation can create look-ahead bias if used carelessly in backtests.

The goal is not to claim that Brownian bridges are the best possible imputation method. The goal is to build a transparent stochastic baseline that connects data cleaning to the probability foundations from Phase 0.

## 1. Why imputation matters in quant research

Financial datasets often contain missing values because of:

- vendor outages;
- market holidays;
- suspended trading;
- stale quotes;
- illiquid instruments;
- corporate actions;
- exchange-specific trading calendars;
- data ingestion failures;
- contract roll gaps in futures data.

A naive response is to call:

```python
series.ffill()
```

or:

```python
series.interpolate()
```

These methods are simple, but they hide uncertainty. They return one completed time series as if the missing values were known exactly.

A Brownian bridge approach is different. It conditions on the observed values before and after a gap and treats the missing path as stochastic.

That means the imputed values are not merely filled values. They are conditional estimates with uncertainty.

## 2. Brownian bridge theory

Let $(X_t)_{t \geq 0}$ follow an arithmetic Brownian motion locally:

$$
X_t = X_0 + \mu t + \sigma W_t
$$

Suppose two endpoints are observed:

$$
X_a = x_a
$$

and:

$$
X_b = x_b
$$

for $a < t < b$.

Conditional on these endpoints, the Brownian bridge distribution at time $t$ is normal:

$$
\begin{aligned}
X_t \mid X_a=x_a, X_b=x_b &\sim \mathcal N\left( x_a + \frac{t-a}{b-a}(x_b-x_a), \sigma^2 \frac{(t-a)(b-t)}{b-a} \right)
\end{aligned}
$$

The conditional mean is simply the straight line between the endpoints:

$$
\begin{aligned}
\mathbb E[X_t \mid X_a, X_b] &= x_a + \frac{t-a}{b-a}(x_b-x_a)
\end{aligned}
$$

The conditional variance is zero at both endpoints and largest in the middle of the gap:

$$
\begin{aligned}
\operatorname{Var}(X_t \mid X_a, X_b) &= \sigma^2 \frac{(t-a)(b-t)}{b-a}
\end{aligned}
$$

In financial price data, it is usually more sensible to apply this to **log prices**:

$$
X_t = \log S_t
$$

because log prices can move on the whole real line, while raw prices must remain positive.

## 3. Important warning: imputation and look-ahead bias

Brownian bridge imputation uses both sides of a missing gap.

For example, to fill missing values between Monday and Friday, it uses Friday's observed price.

That makes it a **smoothing** method, not a real-time forecasting method.

This is acceptable for some ex-post data reconstruction tasks, such as:

- repairing a vendor outage after the fact;
- estimating realised volatility over a known historical period;
- stress-testing sensitivity to missing data;
- creating multiple plausible historical scenarios.

It is dangerous for point-in-time trading signals.

If a strategy at Wednesday uses a Brownian-bridge-imputed Wednesday value that depends on Friday's price, the backtest has leaked future information.

Therefore, every imputed row should be flagged, and downstream notebooks should decide whether to:

1. remove imputed regions;
2. use only causal imputations;
3. run sensitivity analysis;
4. allow bridge imputation only in explicitly ex-post analyses.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

## 4. Synthetic clean price series

We first generate a complete synthetic price series using geometric Brownian motion.

The log-price process is:

$$
\begin{aligned}
\log S_t &= \log S_0 + \left(\mu - \frac{1}{2}\sigma^2\right)t + \sigma W_t
\end{aligned}
$$

The exact model is not the point. We only need a complete ground-truth series so that we can delete observations and measure imputation error.

In [ ]:
def simulate_gbm_daily_prices(
    start: str,
    end: str,
    s0: float = 100.0,
    annual_mu: float = 0.08,
    annual_sigma: float = 0.22,
    seed: int = 42
) -> pd.DataFrame:
    """
    Simulate a business-day geometric Brownian motion price series.

    Returns a DataFrame with timestamp, close, and log_close.
    """
    local_rng = np.random.default_rng(seed)

    dates = pd.date_range(start=start, end=end, freq="B", tz="UTC")
    n = len(dates)

    dt = 1.0 / 252.0
    drift = (annual_mu - 0.5 * annual_sigma ** 2) * dt
    diffusion = annual_sigma * np.sqrt(dt) * local_rng.standard_normal(n)

    log_returns = drift + diffusion
    log_prices = np.log(s0) + np.cumsum(log_returns)
    close = np.exp(log_prices)

    data = pd.DataFrame({
        "symbol": "SYNTH",
        "timestamp": dates,
        "close_true": close,
        "log_close_true": log_prices
    })

    return data


clean_data = simulate_gbm_daily_prices(
    start="2022-01-01",
    end="2024-12-31",
    s0=100.0,
    annual_mu=0.08,
    annual_sigma=0.22,
    seed=42
)

clean_data.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(clean_data["timestamp"], clean_data["close_true"])
plt.title("Complete Synthetic Price Series")
plt.xlabel("Timestamp")
plt.ylabel("Close price")
plt.show()

## 5. Introduce controlled missingness

We create two types of missingness:

1. **Random point missingness** — isolated observations disappear.
2. **Block missingness** — a vendor outage removes consecutive observations.

This lets us evaluate how each imputation method behaves under both small and large gaps.

In [ ]:
def introduce_missing_values(
    data: pd.DataFrame,
    point_missing_fraction: float = 0.03,
    block_starts: list[int] | None = None,
    block_lengths: list[int] | None = None,
    seed: int = 123
) -> pd.DataFrame:
    """
    Introduce random point missingness and block missingness into close prices.
    """
    local_rng = np.random.default_rng(seed)
    out = data.copy()

    n = len(out)
    missing_mask = np.zeros(n, dtype=bool)

    # Avoid first and last rows so Brownian bridge has endpoints for most gaps.
    eligible = np.arange(1, n - 1)
    point_missing_count = int(point_missing_fraction * len(eligible))
    point_indices = local_rng.choice(eligible, size=point_missing_count, replace=False)
    missing_mask[point_indices] = True

    if block_starts is None:
        block_starts = [180, 420, 620]

    if block_lengths is None:
        block_lengths = [8, 15, 25]

    for start, length in zip(block_starts, block_lengths):
        end = min(start + length, n - 1)
        missing_mask[start:end] = True

    out["close_observed"] = out["close_true"]
    out.loc[missing_mask, "close_observed"] = np.nan

    out["is_missing"] = missing_mask
    out["log_close_observed"] = np.log(out["close_observed"])

    return out


missing_data = introduce_missing_values(clean_data)

missing_data[["timestamp", "close_true", "close_observed", "is_missing"]].head(12)

In [ ]:
missing_summary = pd.Series({
    "total_rows": len(missing_data),
    "missing_rows": int(missing_data["is_missing"].sum()),
    "missing_fraction": float(missing_data["is_missing"].mean())
})

missing_summary

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(missing_data["timestamp"], missing_data["close_true"], label="True close", alpha=0.6)
plt.scatter(
    missing_data.loc[missing_data["is_missing"], "timestamp"],
    missing_data.loc[missing_data["is_missing"], "close_true"],
    s=12,
    label="Deleted observations"
)
plt.title("Synthetic Price Series with Deleted Observations")
plt.xlabel("Timestamp")
plt.ylabel("Close price")
plt.legend()
plt.show()

## 6. Baseline imputations

Before using Brownian bridge imputation, we define two simple baselines:

1. **Forward fill**

$$
\hat S_t = S_{t^-}
$$

2. **Linear interpolation in log-price space**

$$
\begin{aligned}
\widehat{\log S_t} &= \log S_a + \frac{t-a}{b-a}(\log S_b - \log S_a)
\end{aligned}
$$

Linear interpolation is equal to the Brownian bridge conditional mean in log-price space.

The difference is that the Brownian bridge framework also gives a conditional variance and allows multiple plausible paths.

In [ ]:
def add_baseline_imputations(data: pd.DataFrame) -> pd.DataFrame:
    """
    Add forward-fill and linear log-price interpolation baselines.
    """
    out = data.copy()

    out["log_close_ffill"] = out["log_close_observed"].ffill()
    out["close_ffill"] = np.exp(out["log_close_ffill"])

    out["log_close_linear"] = out["log_close_observed"].interpolate(
        method="linear",
        limit_area="inside"
    )
    out["close_linear"] = np.exp(out["log_close_linear"])

    return out


imputation_data = add_baseline_imputations(missing_data)

imputation_data[[
    "timestamp",
    "close_true",
    "close_observed",
    "close_ffill",
    "close_linear",
    "is_missing"
]].head(12)

## 7. Estimating local volatility for the bridge

The Brownian bridge variance requires a volatility estimate.

For log prices sampled daily, estimate per-step volatility from observed log returns:

$$
\begin{aligned}
\hat\sigma_{\text{step}} &= \operatorname{Std}(\Delta \log S_t)
\end{aligned}
$$

This is deliberately simple.

In a more advanced system, volatility could be estimated using:

- rolling windows;
- exponentially weighted volatility;
- GARCH;
- realised volatility;
- state-space models;
- stochastic volatility models.

In [ ]:
def estimate_step_volatility(log_price: pd.Series) -> float:
    """
    Estimate per-step volatility from observed log returns.
    """
    observed_log_price = log_price.dropna()
    log_returns = observed_log_price.diff().dropna()
    return float(log_returns.std(ddof=1))


sigma_step = estimate_step_volatility(imputation_data["log_close_observed"])

sigma_step

## 8. Brownian bridge conditional mean and standard deviation

For a gap between observed endpoints $i_0$ and $i_1$, let:

$$
L = i_1 - i_0
$$

For a missing point $j$, where $i_0 < j < i_1$, the conditional mean is:

$$
\begin{aligned}
\mu_j &= x_{i_0} + \frac{j-i_0}{L}(x_{i_1}-x_{i_0})
\end{aligned}
$$

and the conditional variance is:

$$
\begin{aligned}
\sigma_j^2 &= \hat\sigma_{\text{step}}^2 \frac{(j-i_0)(i_1-j)}{L}
\end{aligned}
$$

where $x_i = \log S_i$.

The standard deviation is largest near the middle of the gap and zero at the endpoints.

In [ ]:
def find_missing_blocks(mask: np.ndarray) -> list[tuple[int, int]]:
    """
    Find consecutive True blocks in a boolean mask.

    Returns a list of (start, end) pairs, where end is exclusive.
    """
    blocks = []
    n = len(mask)
    i = 0

    while i < n:
        if not mask[i]:
            i += 1
            continue

        start = i
        while i < n and mask[i]:
            i += 1
        end = i
        blocks.append((start, end))

    return blocks


def brownian_bridge_impute_log_prices(
    log_price_observed: pd.Series,
    sigma_step: float,
    rng: np.random.Generator,
    sample: bool = False
) -> pd.DataFrame:
    """
    Impute missing log prices using Brownian bridge conditional distributions.

    Parameters
    ----------
    log_price_observed:
        Log-price series with missing values.

    sigma_step:
        Per-step volatility estimate.

    rng:
        NumPy random generator.

    sample:
        If False, return the conditional mean.
        If True, sample one stochastic bridge path inside each missing block.

    Returns
    -------
    DataFrame with imputed log price, conditional mean, and conditional standard deviation.
    """
    x = log_price_observed.to_numpy(dtype=float)
    n = len(x)

    imputed = x.copy()
    conditional_mean = np.full(n, np.nan)
    conditional_std = np.full(n, np.nan)

    missing_mask = np.isnan(x)
    blocks = find_missing_blocks(missing_mask)

    for start, end in blocks:
        left = start - 1
        right = end

        # Brownian bridge needs both endpoints.
        if left < 0 or right >= n or np.isnan(x[left]) or np.isnan(x[right]):
            continue

        x_left = x[left]
        x_right = x[right]
        L = right - left

        missing_indices = np.arange(start, end)
        tau = (missing_indices - left) / L

        means = x_left + tau * (x_right - x_left)
        variances = sigma_step ** 2 * ((missing_indices - left) * (right - missing_indices) / L)
        stds = np.sqrt(variances)

        conditional_mean[missing_indices] = means
        conditional_std[missing_indices] = stds

        if sample:
            # A full Brownian bridge path has correlated conditional values.
            # We simulate increments and then condition them to hit the endpoint.
            m = L
            increments = sigma_step * rng.standard_normal(m)
            raw_path = x_left + np.cumsum(increments)
            raw_endpoint = raw_path[-1]

            # Adjust the path linearly so it lands exactly at x_right.
            adjustment = np.linspace(1 / m, 1.0, m) * (x_right - raw_endpoint)
            bridged_path = raw_path + adjustment

            imputed[missing_indices] = bridged_path[:len(missing_indices)]
        else:
            imputed[missing_indices] = means

    return pd.DataFrame({
        "log_close_bridge": imputed,
        "log_close_bridge_mean": np.where(np.isnan(conditional_mean), imputed, conditional_mean),
        "log_close_bridge_std": np.nan_to_num(conditional_std, nan=0.0),
        "bridge_imputed": missing_mask & ~np.isnan(imputed)
    })

In [ ]:
bridge_output = brownian_bridge_impute_log_prices(
    log_price_observed=imputation_data["log_close_observed"],
    sigma_step=sigma_step,
    rng=rng,
    sample=False
)

imputation_data = pd.concat([imputation_data, bridge_output], axis=1)
imputation_data["close_bridge_mean"] = np.exp(imputation_data["log_close_bridge"])

imputation_data[[
    "timestamp",
    "close_true",
    "close_observed",
    "close_ffill",
    "close_linear",
    "close_bridge_mean",
    "is_missing",
    "bridge_imputed"
]].head(12)

## 9. Visualising Brownian bridge uncertainty

The Brownian bridge conditional mean is the line between the observed endpoints in log space.

The conditional standard deviation is smallest near observed endpoints and largest in the middle of a gap.

We visualise one block gap to show this uncertainty explicitly.

In [ ]:
missing_blocks = find_missing_blocks(imputation_data["is_missing"].to_numpy())
missing_blocks[:5]

In [ ]:
# Choose the longest missing block for visualisation.
longest_block = max(missing_blocks, key=lambda block: block[1] - block[0])
start, end = longest_block

window_left = max(0, start - 20)
window_right = min(len(imputation_data), end + 20)
window = imputation_data.iloc[window_left:window_right].copy()

upper_log = window["log_close_bridge_mean"] + 1.96 * window["log_close_bridge_std"]
lower_log = window["log_close_bridge_mean"] - 1.96 * window["log_close_bridge_std"]

plt.figure(figsize=(10, 6))
plt.plot(window["timestamp"], window["close_true"], label="True close", alpha=0.7)
plt.scatter(window["timestamp"], window["close_observed"], label="Observed close", s=18)
plt.plot(window["timestamp"], window["close_bridge_mean"], label="Brownian bridge mean")
plt.fill_between(
    window["timestamp"],
    np.exp(lower_log),
    np.exp(upper_log),
    alpha=0.25,
    label="Approx. 95% bridge interval"
)
plt.title("Brownian Bridge Imputation with Conditional Uncertainty")
plt.xlabel("Timestamp")
plt.ylabel("Close price")
plt.legend()
plt.show()

## 10. Multiple stochastic bridge paths

The conditional mean is only one estimate.

A better way to represent uncertainty is to generate multiple plausible bridge paths.

This is useful for sensitivity analysis:

> Does the downstream signal or risk estimate change materially across plausible imputations?

If yes, the missing-data region should be treated with caution.

In [ ]:
def generate_multiple_bridge_imputations(
    data: pd.DataFrame,
    sigma_step: float,
    num_samples: int,
    seed: int = 999
) -> pd.DataFrame:
    """
    Generate multiple stochastic Brownian bridge imputations.
    """
    local_rng = np.random.default_rng(seed)
    samples = []

    for sample_idx in range(num_samples):
        bridge_sample = brownian_bridge_impute_log_prices(
            log_price_observed=data["log_close_observed"],
            sigma_step=sigma_step,
            rng=local_rng,
            sample=True
        )

        samples.append(np.exp(bridge_sample["log_close_bridge"].to_numpy()))

    sample_columns = [f"bridge_sample_{i:03d}" for i in range(num_samples)]
    return pd.DataFrame(np.column_stack(samples), columns=sample_columns)


bridge_samples = generate_multiple_bridge_imputations(
    data=imputation_data,
    sigma_step=sigma_step,
    num_samples=100,
    seed=999
)

bridge_samples.head()

In [ ]:
window_samples = bridge_samples.iloc[window_left:window_right]

plt.figure(figsize=(10, 6))
plt.plot(window["timestamp"], window["close_true"], label="True close", linewidth=2)
plt.scatter(window["timestamp"], window["close_observed"], label="Observed close", s=18)

for col in bridge_samples.columns[:25]:
    plt.plot(window["timestamp"], window_samples[col], alpha=0.18)

plt.title("Multiple Stochastic Brownian Bridge Imputations")
plt.xlabel("Timestamp")
plt.ylabel("Close price")
plt.legend()
plt.show()

## 11. Error evaluation on deliberately masked values

Because this is synthetic data, we know the true deleted values.

We evaluate imputation error only where observations were deliberately removed.

For each method, compute:

$$
\begin{aligned}
\text{RMSE} &= \sqrt{\frac{1}{m}\sum_{j=1}^{m}(\hat S_j - S_j)^2}
\end{aligned}
$$

and:

$$
\begin{aligned}
\text{MAE} &= \frac{1}{m}\sum_{j=1}^{m}|\hat S_j - S_j|
\end{aligned}
$$

In [ ]:
def evaluate_imputation_errors(data: pd.DataFrame, imputed_columns: list[str]) -> pd.DataFrame:
    """
    Evaluate imputation errors on rows deliberately marked missing.
    """
    eval_data = data[data["is_missing"]].copy()

    rows = []

    for col in imputed_columns:
        valid = eval_data[["close_true", col]].dropna()
        error = valid[col] - valid["close_true"]

        rows.append({
            "method": col,
            "n_evaluated": len(valid),
            "rmse": float(np.sqrt(np.mean(error ** 2))),
            "mae": float(np.mean(np.abs(error))),
            "mean_error": float(np.mean(error)),
            "max_abs_error": float(np.max(np.abs(error)))
        })

    return pd.DataFrame(rows).sort_values("rmse")


error_table = evaluate_imputation_errors(
    imputation_data,
    imputed_columns=["close_ffill", "close_linear", "close_bridge_mean"]
)

error_table

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(error_table["method"], error_table["rmse"])
plt.title("Imputation RMSE on Deliberately Masked Values")
plt.xlabel("Method")
plt.ylabel("RMSE")
plt.xticks(rotation=30, ha="right")
plt.show()

## 12. Coverage check for Brownian bridge intervals

Because the bridge gives a conditional distribution, we can check whether the true deleted values fall inside the approximate 95% interval.

In log-price space, the interval is:

$$
\mu_j \pm 1.96\sigma_j
$$

This is only approximate because:

- volatility is estimated;
- the synthetic data is GBM rather than pure arithmetic Brownian motion in calendar time;
- missingness patterns are artificial;
- finite-sample effects matter.

In [ ]:
coverage_data = imputation_data[imputation_data["is_missing"]].copy()

lower = coverage_data["log_close_bridge_mean"] - 1.96 * coverage_data["log_close_bridge_std"]
upper = coverage_data["log_close_bridge_mean"] + 1.96 * coverage_data["log_close_bridge_std"]
true_log = coverage_data["log_close_true"]

coverage_rate = ((true_log >= lower) & (true_log <= upper)).mean()

coverage_summary = pd.Series({
    "nominal_interval": "95%",
    "empirical_coverage": float(coverage_rate),
    "n_evaluated": int(len(coverage_data))
})

coverage_summary

## 13. Gap-length diagnostics

Imputation quality generally deteriorates as missing gaps become longer.

For short gaps, endpoint conditioning can be helpful.

For long gaps, the bridge distribution becomes wide and the imputed path should be treated as highly uncertain.

In [ ]:
def attach_gap_ids(data: pd.DataFrame, missing_col: str = "is_missing") -> pd.DataFrame:
    """
    Assign a gap ID and gap length to missing blocks.
    """
    out = data.copy()
    out["gap_id"] = pd.NA
    out["gap_length"] = 0

    blocks = find_missing_blocks(out[missing_col].to_numpy())

    for gap_id, (start, end) in enumerate(blocks, start=1):
        out.loc[out.index[start:end], "gap_id"] = gap_id
        out.loc[out.index[start:end], "gap_length"] = end - start

    return out


gap_data = attach_gap_ids(imputation_data)

gap_errors = gap_data[gap_data["is_missing"]].copy()
gap_errors["abs_error_bridge"] = np.abs(gap_errors["close_bridge_mean"] - gap_errors["close_true"])
gap_errors["abs_error_ffill"] = np.abs(gap_errors["close_ffill"] - gap_errors["close_true"])
gap_errors["abs_error_linear"] = np.abs(gap_errors["close_linear"] - gap_errors["close_true"])

by_gap_length = gap_errors.groupby("gap_length")[[
    "abs_error_bridge",
    "abs_error_ffill",
    "abs_error_linear"
]].mean().reset_index()

by_gap_length

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(by_gap_length["gap_length"], by_gap_length["abs_error_bridge"], marker="o", label="Brownian bridge mean")
plt.plot(by_gap_length["gap_length"], by_gap_length["abs_error_ffill"], marker="o", label="Forward fill")
plt.plot(by_gap_length["gap_length"], by_gap_length["abs_error_linear"], marker="o", label="Linear log interpolation")
plt.title("Average Absolute Error by Missing Gap Length")
plt.xlabel("Gap length")
plt.ylabel("Average absolute error")
plt.legend()
plt.show()

## 14. Creating an imputed canonical output

Downstream notebooks should not lose track of which rows were observed and which were imputed.

A useful output should include:

- observed close;
- imputed close;
- final close used by downstream code;
- imputation method;
- imputation uncertainty;
- missing-data flag.

This makes it possible to run sensitivity analysis or exclude imputed sections later.

In [ ]:
def build_imputed_output(data: pd.DataFrame) -> pd.DataFrame:
    """
    Build a clean output table with imputation metadata.
    """
    out = pd.DataFrame({
        "symbol": data["symbol"],
        "timestamp": data["timestamp"],
        "close_true_for_evaluation": data["close_true"],
        "close_observed": data["close_observed"],
        "close_final": data["close_observed"].combine_first(data["close_bridge_mean"]),
        "was_missing": data["is_missing"],
        "was_imputed": data["is_missing"] & data["close_bridge_mean"].notna(),
        "imputation_method": np.where(data["is_missing"], "brownian_bridge_log_price", "observed"),
        "bridge_log_std": data["log_close_bridge_std"]
    })

    return out


imputed_output = build_imputed_output(imputation_data)
imputed_output.head(12)

In [ ]:
output_checks = pd.Series({
    "rows": len(imputed_output),
    "missing_close_final": int(imputed_output["close_final"].isna().sum()),
    "imputed_rows": int(imputed_output["was_imputed"].sum()),
    "positive_close_final": bool((imputed_output["close_final"] > 0).all())
})

output_checks

## 15. Sensitivity of realised volatility to imputation

One reason imputation matters is that downstream diagnostics can change.

For example, realised volatility from close-to-close log returns is:

$$
\begin{aligned}
\hat\sigma_{\text{ann}} &= \sqrt{252}\operatorname{Std}(\Delta \log S_t)
\end{aligned}
$$

We compare realised volatility under:

1. the true complete series;
2. the observed series with missing values dropped;
3. forward fill;
4. linear interpolation;
5. Brownian bridge mean.

In [ ]:
def annualised_realised_vol(close: pd.Series, periods_per_year: int = 252) -> float:
    """
    Compute annualised close-to-close realised volatility.
    """
    log_returns = np.log(close).diff().dropna()
    return float(np.sqrt(periods_per_year) * log_returns.std(ddof=1))


vol_comparison = pd.Series({
    "true_complete": annualised_realised_vol(imputation_data["close_true"]),
    "observed_drop_missing": annualised_realised_vol(imputation_data["close_observed"].dropna()),
    "forward_fill": annualised_realised_vol(imputation_data["close_ffill"]),
    "linear_log_interpolation": annualised_realised_vol(imputation_data["close_linear"]),
    "brownian_bridge_mean": annualised_realised_vol(imputation_data["close_bridge_mean"])
})

vol_comparison

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(vol_comparison.index, vol_comparison.values)
plt.title("Realised Volatility Sensitivity to Imputation Method")
plt.xlabel("Series")
plt.ylabel("Annualised realised volatility")
plt.xticks(rotation=30, ha="right")
plt.show()

## 16. Practical imputation policy

For this repository, a sensible policy is:

| Situation | Suggested action |
|---|---|
| Single missing price in historical diagnostics | Brownian bridge or linear interpolation with flag |
| Short vendor outage in ex-post analysis | Brownian bridge with uncertainty bands and sensitivity checks |
| Missing values inside a live backtest signal window | Avoid two-sided bridge imputation; use causal method or skip |
| Long missing block | Do not trust point imputation; exclude or run scenario analysis |
| Missing leading/trailing values | Brownian bridge cannot apply without both endpoints |
| Illiquid asset with stale prices | Treat as market microstructure problem, not just missing data |
| Futures roll gap | Use contract rolling methodology, not Brownian bridge |
| Options surface gap | Use arbitrage-aware surface interpolation, not simple bridge filling |

The safest rule is:

> Imputation should never silently convert uncertain data into apparently certain data.

## 17. Limitations

This notebook deliberately uses a simple Brownian bridge model.

### 17.1 Brownian bridge assumes continuous paths

Brownian bridges do not model jumps.

If a missing interval contains a news shock, limit move, earnings announcement, or macro release, the bridge may produce a smooth path that never existed.

### 17.2 Volatility is treated as constant inside the gap

The bridge variance uses one estimated per-step volatility.

Real volatility is time-varying and may increase precisely during periods when data is missing.

### 17.3 The method is univariate

This notebook imputes one asset at a time.

Real markets are cross-sectionally linked. If one asset is missing but related assets traded normally, a multivariate method may be better.

### 17.4 It can create look-ahead bias

Because the bridge uses the future endpoint, it is not point-in-time safe for live signal generation.

This must be controlled in backtesting.

### 17.5 It does not fix bad prices

The method fills missing values. It does not detect incorrect observed values.

Bad ticks, split errors, and stale prints require separate data sanitisation.

### 17.6 It does not handle market calendars fully

This notebook uses business-day indices as equal time steps.

A real system should account for asset-specific trading calendars, holidays, early closes, and overnight/weekend effects.

## 18. Research frontier and current directions

Brownian bridge imputation is a transparent stochastic baseline. Modern research and industry practice often move beyond it when data is multivariate, irregular, high-frequency, or structurally constrained.

### 18.1 State-space models and Kalman smoothing

For many financial time series, missing observations can be handled with state-space models.

Kalman filtering is causal, while Kalman smoothing uses information from both sides of a gap.

This distinction mirrors the key warning in this notebook: smoothing is useful for reconstruction, but not automatically safe for point-in-time trading signals.

A future notebook could compare Brownian bridge imputation with a local-level Kalman smoother.

### 18.2 Gaussian process regression

Gaussian processes provide a probabilistic interpolation framework with flexible covariance kernels.

They can represent uncertainty over missing values and can incorporate time-varying smoothness assumptions.

A future notebook could compare Brownian bridge interpolation with a Gaussian process using a Matérn kernel.

### 18.3 Diffusion and bridge models for time series imputation

Modern generative models increasingly use diffusion processes and bridge processes for conditional time-series generation and imputation.

Unlike a simple Brownian bridge, these models can learn complex conditional distributions from data.

A future notebook could implement a small conditional diffusion imputer on synthetic multivariate returns and compare its uncertainty calibration with Brownian bridge sampling.

### 18.4 Flow matching for missing data

Flow matching and continuous normalising flow methods are emerging as alternatives to diffusion-based imputation.

They aim to learn transformations between distributions and can be adapted to conditional missing-data settings.

A future notebook could explore conditional flow matching conceptually before moving to a small implementation.

### 18.5 Multivariate and cross-asset imputation

A missing price for one asset may be predictable from related assets, futures, ETFs, FX rates, or sector indices.

Multivariate imputation methods can use this cross-sectional information, whereas a univariate Brownian bridge ignores it.

A future notebook could use PCA or a dynamic factor model to impute missing observations across a basket of assets.

### 18.6 Point-in-time data engineering

In institutional research, the imputation method is only part of the problem.

The system must also record:

- when the missingness was discovered;
- when the repair was performed;
- what information was available at that time;
- whether downstream models are allowed to use repaired values.

A future notebook could introduce bitemporal data fields for observed time and knowledge time.

## 19. Suggested follow-up notebooks

This notebook naturally leads to:

1. **01_03_continuous_futures_contract_rolling**  
   Handling gaps caused by futures expiries and roll transitions.

2. **01_04_return_sanitization_and_bias_control**  
   Detecting bad returns, split errors, stale prices, and look-ahead bias.

3. **01_05_empirical_distribution_analyzer**  
   Studying how imputation choices affect return distributions and tails.

4. **03_02_har_realized_volatility_forecasting**  
   Understanding how missing data affects realised volatility models.

5. **04_06_var_cvar_and_stress_testing**  
   Testing how missing-data reconstruction changes risk estimates.

6. **05_01_vectorized_backtest_engine**  
   Deciding whether imputed observations are allowed inside a backtest.

7. **07_01_multi_asset_cta_research_pipeline**  
   Running sensitivity analysis across multiple imputation policies.

## 20. Summary

This notebook introduced Brownian bridge imputation as a stochastic method for filling missing financial time-series values.

The main mathematical result was:

$$
\begin{aligned}
X_t \mid X_a=x_a, X_b=x_b &\sim \mathcal N\left( x_a + \frac{t-a}{b-a}(x_b-x_a), \sigma^2 \frac{(t-a)(b-t)}{b-a} \right)
\end{aligned}
$$

The main implementation choices were:

1. impute log prices rather than raw prices;
2. estimate per-step volatility from observed log returns;
3. preserve observed endpoints exactly;
4. return imputation uncertainty, not only filled values;
5. flag every imputed row;
6. compare against forward-fill and linear interpolation baselines;
7. measure downstream sensitivity in realised volatility.

The key computational takeaway is:

> Brownian bridge imputation is easy to implement, but the uncertainty and flags are as important as the imputed values themselves.

The key financial takeaway is:

> Two-sided imputation can be useful for historical reconstruction, but it can create look-ahead bias if used inside point-in-time trading signals.

## 21. Further reading

### Foundations

- Shreve, S. E. *Stochastic Calculus for Finance II: Continuous-Time Models.*
- Øksendal, B. *Stochastic Differential Equations: An Introduction with Applications.*
- Glasserman, P. *Monte Carlo Methods in Financial Engineering.*

### Missing data and financial time series

- Brownian bridge EM methods for covariance estimation with missing cumulative financial data.
- Kalman filtering and smoothing for state-space models.
- Gaussian process regression for probabilistic interpolation.

### Current research directions

- Conditional diffusion models for time-series imputation.
- Diffusion bridge models for time-series generation and forecasting.
- Flow matching for missing-data imputation.
- Multivariate imputation using factor models, attention models, or graph structures.
- Point-in-time and bitemporal data engineering for bias control.